In [8]:
# NumPy 2 e necessario para ler os pickles gerados pela limpeza.
%pip install pandas "numpy>=2,<3" nltk scikit-learn "gensim>=4.4" transformers torch
# Apos atualizar pacotes, reinicie o kernel antes de executar as proximas celulas.


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\berna\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
# No Windows, carregue o PyTorch antes de pandas, NLTK e Gensim.
# Reinicie o kernel antes de executar esta celula se houve falha de DLL.
try:
    import torch
except OSError as erro:
    if getattr(erro, 'winerror', None) == 1114:
        raise RuntimeError(
            'Falha ao inicializar as DLLs do PyTorch. Reinicie o kernel e '
            'execute o notebook desde o inicio para carregar torch primeiro.'
        ) from erro
    raise

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError("Execute a instalacao de NumPy >= 2 na primeira celula e reinicie o kernel antes de ler o pickle.")

raiz_projeto = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / '_DadosLimpos').is_dir()), None
)
if raiz_projeto is None:
    raise RuntimeError('Execute o notebook dentro do repositorio PLN_SteamRecommend.')
caminho_dados = raiz_projeto / '_DadosLimpos' / 'jogos_steam.pkl.gz'
df = pd.read_pickle(caminho_dados)
colunas_esperadas = {
    'appid', 'name', 'genres', 'description',
    'description_idioma', 'description_tokens_sem_stopwords',
}
colunas_ausentes = colunas_esperadas - set(df.columns)
if colunas_ausentes:
    raise ValueError(f'Reexecute a limpeza; faltam as colunas: {sorted(colunas_ausentes)}')
if not df['description_idioma'].eq('en').all():
    raise ValueError('Reexecute a limpeza com o filtro de ingles antes da busca.')
if df.empty:
    raise ValueError('O catalogo limpo esta vazio.')
if not df['description_tokens_sem_stopwords'].map(lambda tokens: isinstance(tokens, list)).all():
    raise ValueError('Os tokens devem ser listas preservadas pelo pickle da limpeza.')

# A posicao de cada jogo sera a mesma em todas as matrizes de embeddings.
df = df.reset_index(drop=True)
print(f'Jogos Steam disponiveis para busca: {len(df):,}')
df[['appid', 'name', 'genres', 'description']].head()


Jogos Steam disponiveis para busca: 178,720


,appid,name,genres,description
0,10,Counter-Strike,Action,Play the world s number 1 online action game E...
1,20,Team Fortress Classic,Action,One of the most popular online action games of...
2,30,Day of Defeat,Action,Enlist in an intense brand of Axis vs Allied t...
3,40,Deathmatch Classic,Action,Enjoy fast paced multiplayer gaming with Death...
4,50,Half-Life: Opposing Force,Action,Return to the Black Mesa Research Facility as ...


In [10]:
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', raise_on_error=True)

# Mesmas etapas da limpeza para representar as consultas em ingles.
tokenizador = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)
apostrofos = str.maketrans({'\u2018': "'", '\u2019': "'", '\u02bc': "'"})

def tokenizar(texto):
    return tokenizador.tokenize(texto)

def normalizar(tokens):
    return [
        unicodedata.normalize(
            'NFKC', unicodedata.normalize('NFKC', token).translate(apostrofos).lower()
        ) for token in tokens
    ]

negacoes = {'no', 'not', 'nor', 'never', 'neither', 'nothing', 'nobody', 'without', 'cannot', "n't"}
stopwords_en = {
    palavra for palavra in normalizar(stopwords.words('english'))
    if palavra not in negacoes and not palavra.endswith("n't")
}

def remover_stopwords(tokens):
    return [token for token in tokens if token not in stopwords_en]

# Reutiliza as listas ja preparadas; nao tokeniza o catalogo novamente.
df['tokens_sem_stopwords'] = df['description_tokens_sem_stopwords']
df[['appid', 'name', 'tokens_sem_stopwords']].head()


,appid,name,tokens_sem_stopwords
0,10,Counter-Strike,"[play, world, number, 1, online, action, game,..."
1,20,Team Fortress Classic,"[one, popular, online, action, games, time, te..."
2,30,Day of Defeat,"[enlist, intense, brand, axis, vs, allied, tea..."
3,40,Deathmatch Classic,"[enjoy, fast, paced, multiplayer, gaming, deat..."
4,50,Half-Life: Opposing Force,"[return, black, mesa, research, facility, one,..."


# Representação 1 - Word2Vec

In [11]:
from gensim.models import Word2Vec

# Treinar Word2Vec nas descricoes dos jogos Steam.
modelo_w2v = Word2Vec(
    sentences=df["tokens_sem_stopwords"].tolist(),
    vector_size=190,
    window=5,
    min_count=1,
    sg=1,          # Skip-gram
    seed=42,
    workers=1,     # ajuda na reprodutibilidade
    epochs=10
)

print("Tamanho do vocabulário:", len(modelo_w2v.wv))
print("Dimensões dos vetores:", modelo_w2v.vector_size)

# Algumas palavras disponíveis no vocabulário
list(modelo_w2v.wv.index_to_key)[:20]


Tamanho do vocabulário: 97647
Dimensões dos vetores: 190


['game',
 'world',
 'adventure',
 'play',
 'explore',
 'action',
 'new',
 'build',
 'time',
 'based',
 'find',
 'fight',
 'way',
 'one',
 'survive',
 'puzzle',
 'every',
 'story',
 'players',
 'enemies']

In [12]:
import numpy as np

def vetor_documento_word2vec(tokens):
    # Calcula a média dos vetores Word2Vec das palavras conhecidas.
    vetores = [
        modelo_w2v.wv[token]
        for token in tokens
        if token in modelo_w2v.wv
    ]

    if not vetores:
        return np.zeros(
            modelo_w2v.vector_size,
            dtype=np.float32
        )

    return np.mean(vetores, axis=0)


# Criar uma matriz: uma linha por descrição
embeddings_word2vec = np.vstack(
    df["tokens_sem_stopwords"].apply(
        vetor_documento_word2vec
    )
)

# DataFrame apenas para inspecionar a representação
df_word2vec = pd.DataFrame(
    embeddings_word2vec,
    columns=[
        f"w2v_{i:03d}"
        for i in range(
            embeddings_word2vec.shape[1]
        )
    ]
)

df_word2vec.insert(
    0,
    "appid",
    df["appid"].values
)

print(
    "Formato da matriz Word2Vec:",
    embeddings_word2vec.shape
)

# Mostrar somente algumas dimensões
df_word2vec.iloc[:5, :8]


Formato da matriz Word2Vec: (178720, 190)


,appid,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006
0,10,0.204895,0.117850,0.073719,-0.026350,-0.137101,0.329192,-0.151584
1,20,0.116498,0.109501,0.106506,-0.046189,-0.057746,0.317678,-0.155836
2,30,0.155569,0.057073,-0.018256,-0.138521,0.020948,0.321817,-0.237794
3,40,0.232176,0.210333,0.145649,0.000174,-0.109954,0.267099,-0.455526
4,50,0.177997,0.094175,0.039929,0.031535,-0.096041,0.255905,-0.200020


# Representacao 2 - BERT em ingles

Usa `google-bert/bert-base-uncased` com media dos tokens (mean pooling). O primeiro carregamento baixa os pesos do modelo. O processamento do catalogo inteiro pode demorar, principalmente em CPU. Este BERT nao foi ajustado especificamente para similaridade de frases; compare os resultados com Word2Vec.


In [13]:
import torch

from transformers import AutoTokenizer, AutoModel

MODELO_BERT = "google-bert/bert-base-uncased"

# Detectar automaticamente GPU, quando disponível
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

# Carregar tokenizador e modelo
tokenizer_bert = AutoTokenizer.from_pretrained(
    MODELO_BERT,
    do_lower_case=True
)

modelo_bert = AutoModel.from_pretrained(
    MODELO_BERT
)

modelo_bert = modelo_bert.to(device)
modelo_bert.eval()

print("Modelo carregado:", MODELO_BERT)


Dispositivo: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 812.85it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado: google-bert/bert-base-uncased


In [14]:
def mean_pooling(
    last_hidden_state,
    attention_mask
):
    # Calcula a média dos vetores dos tokens,
    # ignorando posições utilizadas apenas como padding.
    mascara = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    soma = torch.sum(
        last_hidden_state * mascara,
        dim=1
    )

    quantidade = torch.clamp(
        mascara.sum(dim=1),
        min=1e-9
    )

    return soma / quantidade


def gerar_embeddings_bert(
    textos,
    batch_size=8
):
    # Gera um vetor por texto usando
    # BERT em ingles + mean pooling.
    todos_embeddings = np.empty((len(textos), modelo_bert.config.hidden_size), dtype=np.float32)

    for inicio in range(
        0,
        len(textos),
        batch_size
    ):
        lote = textos[
            inicio:inicio + batch_size
        ]

        entradas = tokenizer_bert(
            lote,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        entradas = {
            chave: valor.to(device)
            for chave, valor
            in entradas.items()
        }

        with torch.no_grad():
            saida = modelo_bert(**entradas)

        embeddings_lote = mean_pooling(
            saida.last_hidden_state,
            entradas["attention_mask"]
        )

        todos_embeddings[inicio:inicio + len(lote)] = embeddings_lote.cpu().numpy()
        if inicio == 0 or (inicio // batch_size + 1) % 100 == 0 or inicio + len(lote) == len(textos):
            print(f'BERT: {inicio + len(lote):,}/{len(textos):,} descricoes', flush=True)

    return todos_embeddings


# Usamos a descricao limpa em ingles, preservando stopwords e contexto.
textos = df["description"].tolist()

embeddings_bert = gerar_embeddings_bert(
    textos
)

print(
    "Formato da matriz BERT:",
    embeddings_bert.shape
)


BERT: 8/178,720 descricoes
BERT: 800/178,720 descricoes
BERT: 1,600/178,720 descricoes
BERT: 2,400/178,720 descricoes
BERT: 3,200/178,720 descricoes
BERT: 4,000/178,720 descricoes
BERT: 4,800/178,720 descricoes
BERT: 5,600/178,720 descricoes
BERT: 6,400/178,720 descricoes
BERT: 7,200/178,720 descricoes
BERT: 8,000/178,720 descricoes
BERT: 8,800/178,720 descricoes
BERT: 9,600/178,720 descricoes
BERT: 10,400/178,720 descricoes
BERT: 11,200/178,720 descricoes
BERT: 12,000/178,720 descricoes
BERT: 12,800/178,720 descricoes
BERT: 13,600/178,720 descricoes
BERT: 14,400/178,720 descricoes
BERT: 15,200/178,720 descricoes
BERT: 16,000/178,720 descricoes
BERT: 16,800/178,720 descricoes
BERT: 17,600/178,720 descricoes
BERT: 18,400/178,720 descricoes
BERT: 19,200/178,720 descricoes
BERT: 20,000/178,720 descricoes
BERT: 20,800/178,720 descricoes
BERT: 21,600/178,720 descricoes
BERT: 22,400/178,720 descricoes
BERT: 23,200/178,720 descricoes
BERT: 24,000/178,720 descricoes
BERT: 24,800/178,720 descri

# Busca

In [15]:
from sklearn.metrics.pairwise import cosine_similarity


def buscar_top_k(vetor_consulta, matriz_documentos, k=10):
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError('k deve ser um inteiro positivo.')
    if matriz_documentos.shape[0] != len(df):
        raise ValueError('As linhas dos embeddings devem corresponder aos jogos de df.')
    vetor_consulta = np.asarray(vetor_consulta).reshape(1, -1)
    if not np.isfinite(vetor_consulta).all() or not np.any(vetor_consulta):
        raise ValueError('A consulta nao possui um vetor valido para busca.')
    similaridades = cosine_similarity(vetor_consulta, matriz_documentos)[0]
    # Vetores nulos nao representam descricoes e ficam fora do ranking.
    validos = np.flatnonzero(np.any(matriz_documentos != 0, axis=1))
    indices = validos[np.argsort(-similaridades[validos], kind='stable')[:k]]
    resultado = df.iloc[indices][['appid', 'name', 'genres', 'description']].copy()
    resultado.insert(0, 'posicao', range(1, len(resultado) + 1))
    resultado['similaridade'] = similaridades[indices]
    resultado['pagina_steam'] = 'https://store.steampowered.com/app/' + resultado['appid'].astype(str)
    return resultado


In [16]:
def representar_consulta_word2vec(
    texto
):
    tokens = tokenizar(texto)
    tokens = normalizar(tokens)
    tokens = remover_stopwords(tokens)

    conhecidos = [
        token
        for token in tokens
        if token in modelo_w2v.wv
    ]

    oov = [
        token
        for token in tokens
        if token not in modelo_w2v.wv
    ]

    if not conhecidos:
        return None, conhecidos, oov

    vetor = np.mean(
        [
            modelo_w2v.wv[token]
            for token in conhecidos
        ],
        axis=0
    )

    return vetor, conhecidos, oov


In [17]:
def representar_consulta_bert(
    texto
):
    return gerar_embeddings_bert(
        [texto]
    )[0]


# Consultar jogos Steam

Execute as representacoes acima e altere a consulta em ingles. Cada modelo compara uma consulta com o catalogo, sem criar uma matriz de todos os pares.


In [18]:
consulta = 'Explore an open world and build a base to survive with friends'
top_k = 10
if not consulta.strip():
    raise ValueError('Digite uma consulta em ingles.')

vetor_w2v, conhecidos, desconhecidos = representar_consulta_word2vec(consulta)
print('Consulta:', consulta)
print('Word2Vec - termos conhecidos:', conhecidos)
print('Word2Vec - termos fora do vocabulario:', desconhecidos)
if vetor_w2v is None:
    print('Word2Vec: nenhum termo conhecido. Tente outras palavras em ingles.')
else:
    resultados_word2vec = buscar_top_k(vetor_w2v, embeddings_word2vec, k=top_k)
    display(resultados_word2vec)

print('Resultados BERT:')
vetor_bert = representar_consulta_bert(consulta)
resultados_bert = buscar_top_k(vetor_bert, embeddings_bert, k=top_k)
display(resultados_bert)


Consulta: Explore an open world and build a base to survive with friends
Word2Vec - termos conhecidos: ['explore', 'open', 'world', 'build', 'base', 'survive', 'friends']
Word2Vec - termos fora do vocabulario: []


,posicao,appid,name,genres,description,similaridade,pagina_steam
69133,1,2141620,Landiscape,Action; Adventure; Indie; Strategy; Free To Play,Landiscape is a relaxed 1 4 player open world ...,0.939624,https://store.steampowered.com/app/2141620
14463,2,655780,Project 5: Sightseer,Action; Adventure; Indie; RPG,Open world multiplayer sandbox RPG game Explor...,0.937014,https://store.steampowered.com/app/655780
117378,3,3435890,Isle of Survival: The New Dawn,Action; Adventure; RPG,Survive and rebuild on a mysterious island Exp...,0.936602,https://store.steampowered.com/app/3435890
75663,4,2313330,TerraTech Worlds,Action; Adventure; Indie; Simulation; Strategy...,TerraTech Worlds is an open world build craft ...,0.933403,https://store.steampowered.com/app/2313330
96253,5,2868100,Shamania,Action; Adventure; RPG,Shamania is an open world survival game where ...,0.933188,https://store.steampowered.com/app/2868100
157209,6,4562650,Island of the Lost,Action; Adventure; Casual; Indie; Simulation,Survive on a mysterious island gather resource...,0.932924,https://store.steampowered.com/app/4562650
39331,7,1337910,Dead Earth Zombies,Action; Adventure; Casual; Indie; RPG; Strateg...,Dead Earth is an open world survival game choo...,0.932216,https://store.steampowered.com/app/1337910
16853,8,717790,Hold Your Own,Action; Adventure; Casual; Indie; Simulation; ...,A single player open world sandbox crafting an...,0.931480,https://store.steampowered.com/app/717790
57190,9,1805750,First Men´s Life,Adventure; Casual; Indie,Create your own world Build fight farm and exp...,0.928565,https://store.steampowered.com/app/1805750
133736,10,3891900,FLINT,Massively Multiplayer,Multiplayer survival in an open world Build yo...,0.928546,https://store.steampowered.com/app/3891900


Resultados BERT:
BERT: 1/1 descricoes


,posicao,appid,name,genres,description,similaridade,pagina_steam
157209,1,4562650,Island of the Lost,Action; Adventure; Casual; Indie; Simulation,Survive on a mysterious island gather resource...,0.857490,https://store.steampowered.com/app/4562650
57190,2,1805750,First Men´s Life,Adventure; Casual; Indie,Create your own world Build fight farm and exp...,0.854418,https://store.steampowered.com/app/1805750
32007,3,1141030,Last Message,Adventure; Casual; Simulation; Free To Play,Use an in game computer to chat with friends t...,0.853731,https://store.steampowered.com/app/1141030
58419,4,1837280,Lost Island,Adventure; Indie; Simulation,Trapped on an Island Find items to survive and...,0.851760,https://store.steampowered.com/app/1837280
59179,5,1857520,Team Ice Cream VR,Casual; Indie; Simulation,Play alone in VR or get together with a friend...,0.848383,https://store.steampowered.com/app/1857520
88691,6,2660460,Aviassembly,Simulation; Early Access,Build and fly your own airplane Complete missi...,0.845177,https://store.steampowered.com/app/2660460
28480,7,1047920,White Sun,RPG,An RPG about surviving a dangerous world with ...,0.845138,https://store.steampowered.com/app/1047920
87857,8,2638240,Escape to the Italian brainrot,Action; Casual; Indie,Explore a strange village filled with Italian ...,0.842539,https://store.steampowered.com/app/2638240
47559,9,1551700,Fractium,Action; Adventure; Strategy,Conquer build and raid in a massive open world...,0.842072,https://store.steampowered.com/app/1551700
74663,10,2286240,Polyturned,Action; Adventure; Indie; Massively Multiplaye...,Your goal is to survive in a city overrun by z...,0.841792,https://store.steampowered.com/app/2286240
